# Feature Engineering
Computes features for each selected pair over the full price history.

**Hedge ratios (evec0/evec1) were estimated on in-sample data in notebook 02 and are frozen here.**
Rolling statistics use backward-looking windows — no lookahead.

In [1]:
import sys, numpy as np, pandas as pd
from statsmodels.tsa.stattools import adfuller
sys.path.append('..')
from src.feature_engineering import hurst_exponent

In [2]:
# ── IN-SAMPLE / OUT-OF-SAMPLE BOUNDARY ───────────────────────────────────────
IS_END         = '2023-12-31'
OOS_START      = '2024-01-01'
ROLLING_WINDOW = 252
# ─────────────────────────────────────────────────────────────────────────────

In [3]:
selected_pairs = pd.read_csv('../data_processed/selected_pairs.csv', index_col=0)
prices_all     = pd.read_csv('../data_raw/prices.csv', index_col=0, parse_dates=True).dropna(how='all')
print(f'Price data: {prices_all.shape[1]} tickers, {prices_all.shape[0]} trading days')
print(f'Date range: {prices_all.index[0].date()} to {prices_all.index[-1].date()}')
print(f'Processing {len(selected_pairs)} pairs...')

Price data: 40 tickers, 1508 trading days
Date range: 2020-01-02 to 2025-12-31
Processing 15 pairs...


In [4]:
for _, row in selected_pairs.iterrows():
    t1, t2         = row['ticker1'], row['ticker2']
    evec0, evec1   = row['evec0'], row['evec1']

    aligned        = pd.DataFrame({t1: prices_all[t1], t2: prices_all[t2]}).dropna()
    log_p1         = np.log(aligned[t1])
    log_p2         = np.log(aligned[t2])

    # Spread using IS-estimated Johansen hedge ratios (frozen; no OOS data used)
    spread         = evec0 * log_p1 + evec1 * log_p2

    # Rolling z-score (252-day backward window)
    spr_mean       = spread.rolling(ROLLING_WINDOW).mean()
    spr_std        = spread.rolling(ROLLING_WINDOW).std()
    z_score        = (spread - spr_mean) / spr_std

    # Rolling ADF p-value (backward window) — ~2 min total for 15 pairs
    adf_vals       = [np.nan] * ROLLING_WINDOW
    for i in range(ROLLING_WINDOW, len(spread)):
        adf_vals.append(adfuller(spread.iloc[i - ROLLING_WINDOW : i])[1])
    adf_series     = pd.Series(adf_vals, index=spread.index, name='adf_p_252')

    # Rolling Hurst exponent (backward window)
    hurst_vals     = [np.nan] * ROLLING_WINDOW
    for i in range(ROLLING_WINDOW, len(spread)):
        hurst_vals.append(hurst_exponent(spread.iloc[i - ROLLING_WINDOW : i].values, max_lag=50))
    hurst_series   = pd.Series(hurst_vals, index=spread.index, name='hurst_252')

    # Spread return (log-return of linear combination; used in backtest, not ML)
    spread_ret     = (evec0 * log_p1.diff() + evec1 * log_p2.diff()).rename('spread_ret')

    features = pd.DataFrame({
        'z_score':       z_score,
        'spread_change': spread.diff(),
        'spread_vol_20': spread.rolling(20).std(),
        'spread_ret':    spread_ret,
        'adf_p_252':     adf_series,
        'hurst_252':     hurst_series,
    })
    features['adf_stationary'] = (features['adf_p_252'] < 0.05).astype(int)
    features['hurst_mr']       = (features['hurst_252'] < 0.5).astype(int)
    features['regime_score']   = features['adf_stationary'] + features['hurst_mr']

    features = features.dropna()
    features.to_csv(f'../data_processed/features_{t1}_{t2}.csv')
    oos_n = (features.index >= OOS_START).sum()
    print(f'  {t1}/{t2}: {len(features)} rows | IS: {len(features) - oos_n} | OOS: {oos_n}')

print('Feature engineering complete.')

  EWJ/HYG: 1256 rows | IS: 754 | OOS: 502
  HYG/XLC: 1256 rows | IS: 754 | OOS: 502
  XLU/XOP: 1256 rows | IS: 754 | OOS: 502
  HYG/MTUM: 1256 rows | IS: 754 | OOS: 502
  EWC/IWD: 1256 rows | IS: 754 | OOS: 502
  GDXJ/LQD: 1256 rows | IS: 754 | OOS: 502
  USO/XLU: 1256 rows | IS: 754 | OOS: 502
  EWG/HYG: 1256 rows | IS: 754 | OOS: 502
  EWJ/XLC: 1256 rows | IS: 754 | OOS: 502
  EWC/XLF: 1256 rows | IS: 754 | OOS: 502
  GDXJ/USO: 1256 rows | IS: 754 | OOS: 502
  XLP/XLV: 1256 rows | IS: 754 | OOS: 502
  XLP/XLU: 1256 rows | IS: 754 | OOS: 502
  EWA/XLB: 1256 rows | IS: 754 | OOS: 502
  GDX/USO: 1256 rows | IS: 754 | OOS: 502
Feature engineering complete.
